# Ex1 — Hypergradient computation
Compare the explicit cross-Jacobian and adjoint–VJP at the same primal–dual MHE solutions.

In [ ]:
from utils import configure_runtime
configure_runtime()
from datetime import datetime
import numpy as np
import torch
import json
import pandas as pd

from data import load_data
from models import (
    MLP_Residual, NeuralForceModel, NeuralForceModelCasadi, set_arrival_initialization,
)
from estimation import ObserverMHEAcados, ekf_warmstart_windows, make_mhe_config, tensor_parameters
from gradients import gradient_adjoint_vjp, gradient_explicit
from utils import ROOT, load_initialization, timed_gradient_call


## 1. Configuration

In [ ]:
RUN_BENCHMARK = False
SEED = 7
N_POINTS = 100
WIDTHS = [4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
INPUT_SCALE = [3.6, 2.1, 19.5, .56, 1.8]
system = dict(nx=3, nu=2, nw=3, N=40, T=0.02,
              E=np.eye(3), C=np.array([[1., 0., 0.], [0., 0., 1.]]),
              integration="euler", rk4_steps=1, m=0.041, l1=0.029, l2=0.033)
system["casadi_model"] = NeuralForceModelCasadi
REFERENCE_DIR = ROOT / "runs" / "Ex1"
OUTPUT_DIR = ROOT / "runs" / "new" / ("Ex1_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

system["hidden_limit"] = 2.5


## 2. Fixed model and arrival weights

In [ ]:
metadata = json.loads((REFERENCE_DIR / "metadata.json").read_text())
initialization = load_initialization("Ex2_PEM-MHE")
for field, saved in (("parameter_initial", "fixed_physics_params"), ("mhe_sbar", "fixed_sbar"),
                     ("mhe_Sigmainv", "fixed_sigmainv"), ("mhe_Qinv", "fixed_qinv"), ("mhe_Rinv", "fixed_rinv")):
    initialization[field] = np.asarray(metadata[saved])

def make_model(width):
    torch.manual_seed(SEED + width)
    network = MLP_Residual(5, 4, [0.]*5, INPUT_SCALE, hidden_dim=width)
    box = dict(lower=initialization["parameter_lower"], upper=initialization["parameter_upper"],
               nominal=initialization["parameter_initial"])
    model = NeuralForceModel(nn_residual=network, use_nn_flag=True, physics_parameter_box=box,
        physics_param_init=initialization["parameter_initial"], sample_time=system["T"], integration_method=system["integration"])
    set_arrival_initialization(model, initialization["mhe_sbar"], initialization["mhe_Sigmainv"])
    model.requires_grad_(False)
    network.requires_grad_(True)
    return model


## 3. Shared MHE solutions

In [ ]:
if RUN_BENCHMARK:
    data = load_data(2)["train"]
    indices = np.linspace(0, len(data["u"])-1, N_POINTS, dtype=int)
    windows = {key: value[indices] for key, value in data.items()}
    model = make_model(4)
    config = make_mhe_config(system, model, initialization, batch_size=1)
    params = tensor_parameters(config)
    observer = ObserverMHEAcados(config)
    seeds = ekf_warmstart_windows(model, windows)
    points = []
    for i in range(N_POINTS):
        xs, ws, mus = observer.estimate_batch(windows["u"][i:i+1], windows["y"][i:i+1],
                                             train_flag=True, warmstart_xs=seeds[i:i+1])
        arrays = (xs[0], ws[0], mus[0], windows["u"][i], windows["y"][i])
        points.append(tuple(torch.as_tensor(a, dtype=torch.float32) for a in arrays))


## 4. Explicit sensitivity and adjoint–VJP
Both paths use the same Gauss–Newton KKT blocks. Time only gradient evaluation.

In [ ]:
if RUN_BENCHMARK:
    OUTPUT_DIR.mkdir(parents=True)
    records = []
    for width in WIDTHS:
        model = make_model(width)
        gradient_adjoint_vjp(params, model, *points[0])
        gradient_explicit(params, model, *points[0])
        n_params = sum(p.numel() for p in model.nn_residual.parameters())
        for i, point in enumerate(points):
            calls = [lambda: gradient_adjoint_vjp(params, model, *point),
                     lambda: gradient_explicit(params, model, *point)]
            timings, outputs = [None, None], [None, None]
            for method in ([0, 1] if i % 2 == 0 else [1, 0]):
                timings[method], outputs[method] = timed_gradient_call(calls[method])
            ours, explicit = outputs[0][1], outputs[1][1]
            error = (torch.linalg.vector_norm(ours-explicit) / torch.linalg.vector_norm(explicit).clamp_min(1e-12)).item()
            assert error < 1e-4, f"Relative gradient error: {error:.3e}"
            records.append((width, n_params, indices[i], *timings, error))
        print(width, n_params, np.median([r[3:5] for r in records if r[0] == width], axis=0))
        fields = ("width", "nn_params", "dataset_index", "ours_ms", "leo_ms", "relative_gradient_error")
        np.savez_compressed(OUTPUT_DIR / "point_results.npz", **dict(zip(fields, np.asarray(records).T)))
    (OUTPUT_DIR / "metadata.json").write_text(json.dumps(dict(metadata,
        n_points_per_completed_width=N_POINTS, completed_widths=WIDTHS), indent=2))


## 5. Runtime summary

In [ ]:
directory = OUTPUT_DIR if RUN_BENCHMARK else REFERENCE_DIR
with np.load(directory / "point_results.npz") as saved:
    results = pd.DataFrame({key: saved[key] for key in saved.files})
results.groupby("nn_params")[["ours_ms", "leo_ms", "relative_gradient_error"]].median()


,ours_ms,leo_ms,relative_gradient_error
nn_params,,,
64,7.825667,10.366979,1.315654e-07
156,7.934062,10.797167,1.354497e-07
436,7.931063,11.435208,1.336066e-07
1380,8.006146,14.187354,1.534601e-07
4804,8.310437,21.575354,1.557453e-07
17796,8.913563,41.560145,1.595480e-07
68356,9.659792,110.185250,1.632938e-07
267780,10.647771,336.759855,1.721882e-07
1059844,14.151999,1296.040479,1.635068e-07
